## 📦 Step 1: Install Required Dependencies

In [ ]:
!pip install kagglehub segmentation-models-pytorch albumentations opencv-python-headless matplotlib seaborn -q
print("✅ All dependencies installed successfully!")

## 📥 Step 2: Download Dataset from Kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ayushcl/severstal-steel-defect-detection")
print("Path to dataset files:", path)

## 🔧 Step 3: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Deep Learning Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models

# Albumentation for augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Sklearn
from sklearn.model_selection import train_test_split

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 📊 Step 4: Load and Explore Data

In [ ]:
# Construct the full path to train.csv
train_csv_path = os.path.join(path, 'train.csv')

# Load the train.csv file into a pandas DataFrame
train_df = pd.read_csv(train_csv_path)

print("📋 First 5 rows of train_df:")
print(train_df.head())

print("\n📊 DataFrame Information:")
train_df.info()

print("\n🔍 DataFrame Shape:", train_df.shape)

In [ ]:
# Check for missing values
print("❓ Missing values in train_df:")
print(train_df.isnull().sum())

# Check unique classes
print("\n🏷️  Unique ClassId values:", train_df['ClassId'].unique() if 'ClassId' in train_df.columns else "ClassId column not found")

In [ ]:
# Explore image directories
train_images_path = os.path.join(path, 'train_images')
test_images_path = os.path.join(path, 'test_images')

train_images_files = os.listdir(train_images_path)
test_images_files = os.listdir(test_images_path)

print(f"📁 First 5 files in 'train_images': {train_images_files[:5]}")
print(f"📁 Total files in 'train_images': {len(train_images_files)}")
print(f"\n📁 First 5 files in 'test_images': {test_images_files[:5]}")
print(f"📁 Total files in 'test_images': {len(test_images_files)}")

## 🎨 Step 5: Data Preprocessing & RLE Decoding

In [ ]:
def rle_to_mask(rle_string, height=256, width=1600):
    """
    Convert RLE (Run-Length Encoding) string to binary mask
    
    Args:
        rle_string: RLE encoded string
        height: Image height
        width: Image width
    
    Returns:
        Binary mask as numpy array
    """
    if pd.isna(rle_string):
        return np.zeros((height, width), dtype=np.uint8)
    
    mask = np.zeros(height * width, dtype=np.uint8)
    rle_numbers = [int(num) for num in rle_string.split()]
    
    for i in range(0, len(rle_numbers), 2):
        start = rle_numbers[i] - 1
        length = rle_numbers[i + 1]
        mask[start:start + length] = 1
    
    mask = mask.reshape((width, height)).T
    return mask

def mask_to_rle(mask):
    """
    Convert binary mask to RLE string
    
    Args:
        mask: Binary mask as numpy array
    
    Returns:
        RLE encoded string
    """
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

print("✅ RLE encoding/decoding functions defined!")

In [ ]:
# First, let's check the actual structure of the CSV
print("🔍 Checking CSV structure...")
print(f"Columns: {train_df.columns.tolist()}")
print(f"\nSample data:")
print(train_df.head(10))

In [ ]:
# Prepare dataframe with image IDs and masks for each class
def prepare_dataframe(df):
    """
    Reorganize dataframe to have one row per image with separate columns for each class mask
    Handles both 'ImageId_ClassId' and separate column formats
    """
    df_dict = {}
    
    # Check if 'ImageId_ClassId' column exists (original Kaggle format)
    if 'ImageId_ClassId' in df.columns:
        print("✅ Found 'ImageId_ClassId' column format")
        for idx, row in df.iterrows():
            image_id = row['ImageId_ClassId'].split('_')[0]
            class_id = row['ImageId_ClassId'].split('_')[1]
            
            if image_id not in df_dict:
                df_dict[image_id] = {'ImageId': image_id, 'Class1': np.nan, 'Class2': np.nan, 
                                      'Class3': np.nan, 'Class4': np.nan}
            
            df_dict[image_id][f'Class{class_id}'] = row['EncodedPixels']
    
    # Alternative format: separate ImageId and ClassId columns
    elif 'ImageId' in df.columns and 'ClassId' in df.columns:
        print("✅ Found separate 'ImageId' and 'ClassId' columns")
        for idx, row in df.iterrows():
            image_id = row['ImageId']
            class_id = str(row['ClassId'])
            
            if image_id not in df_dict:
                df_dict[image_id] = {'ImageId': image_id, 'Class1': np.nan, 'Class2': np.nan, 
                                      'Class3': np.nan, 'Class4': np.nan}
            
            df_dict[image_id][f'Class{class_id}'] = row['EncodedPixels']
    
    # Alternative format: one row per image with class columns already separated
    elif 'ImageId' in df.columns and any(col.startswith('Class') for col in df.columns):
        print("✅ Found already separated class columns")
        return df  # Data is already in the right format
    
    else:
        raise ValueError(f"Unknown CSV format. Columns found: {df.columns.tolist()}")
    
    new_df = pd.DataFrame.from_dict(df_dict, orient='index')
    new_df.reset_index(drop=True, inplace=True)
    
    # Add binary columns for defect presence
    new_df['HasDefect'] = ~(new_df['Class1'].isna() & new_df['Class2'].isna() & 
                            new_df['Class3'].isna() & new_df['Class4'].isna())
    
    return new_df

try:
    train_df_processed = prepare_dataframe(train_df)
    print("\n✅ DataFrame prepared successfully!")
    print(f"📊 Shape: {train_df_processed.shape}")
    print("\n📋 Sample data:")
    print(train_df_processed.head())
    
    # Check defect distribution
    print(f"\n🔍 Images with defects: {train_df_processed['HasDefect'].sum()}")
    print(f"🔍 Images without defects: {(~train_df_processed['HasDefect']).sum()}")
except Exception as e:
    print(f"❌ Error processing dataframe: {e}")
    print(f"Please check the CSV structure above and report the issue.")

## 🖼️ Step 6: Visualize Sample Images with Masks

In [ ]:
def visualize_image_with_masks(image_id, df, images_path, figsize=(20, 5)):
    """
    Visualize image with its defect masks for all 4 classes
    """
    fig, axes = plt.subplots(1, 6, figsize=figsize)
    
    # Load original image
    img_path = os.path.join(images_path, image_id)
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Display original image
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Get row for this image
    row = df[df['ImageId'] == image_id].iloc[0]
    
    # Display masks for each class
    colors = ['red', 'green', 'blue', 'yellow']
    combined_mask = np.zeros_like(image)
    
    for i in range(1, 5):
        mask = rle_to_mask(row[f'Class{i}'])
        axes[i].imshow(mask, cmap='gray')
        axes[i].set_title(f'Class {i} Mask')
        axes[i].axis('off')
        
        # Add to combined mask with different colors
        if mask.sum() > 0:
            color_mask = np.zeros_like(image)
            if colors[i-1] == 'red':
                color_mask[:, :, 0] = mask * 255
            elif colors[i-1] == 'green':
                color_mask[:, :, 1] = mask * 255
            elif colors[i-1] == 'blue':
                color_mask[:, :, 2] = mask * 255
            elif colors[i-1] == 'yellow':
                color_mask[:, :, 0] = mask * 255
                color_mask[:, :, 1] = mask * 255
            
            combined_mask = np.maximum(combined_mask, color_mask)
    
    # Display combined overlay
    overlay = cv2.addWeighted(image, 0.7, combined_mask.astype(np.uint8), 0.3, 0)
    axes[5].imshow(overlay)
    axes[5].set_title('All Defects Overlay')
    axes[5].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize a few samples with defects
defective_images = train_df_processed[train_df_processed['HasDefect'] == True]['ImageId'].values[:3]

print("🖼️  Visualizing sample images with defects:")
for img_id in defective_images:
    visualize_image_with_masks(img_id, train_df_processed, train_images_path)

## 🔨 Step 7: Create Custom Dataset

In [ ]:
class SteelDefectDataset(Dataset):
    """
    Custom Dataset for Steel Defect Detection
    """
    def __init__(self, df, images_path, transform=None, img_size=(256, 256)):
        self.df = df
        self.images_path = images_path
        self.transform = transform
        self.img_size = img_size
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['ImageId']
        
        # Load image
        img_path = os.path.join(self.images_path, image_id)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Create combined mask (all 4 classes)
        mask = np.zeros((256, 1600, 4), dtype=np.float32)
        for i in range(1, 5):
            mask[:, :, i-1] = rle_to_mask(row[f'Class{i}'])
        
        # Apply transformations
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        # Transpose mask to (C, H, W) for PyTorch
        mask = mask.permute(2, 0, 1)
        
        return image, mask, image_id

print("✅ Custom Dataset class created!")

## 🎯 Step 8: Define Data Augmentation

In [ ]:
# Training augmentation
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Validation augmentation (no random augmentation, only resize and normalize)
val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print("✅ Data augmentation pipelines defined!")

## 🧠 Step 9: Build ResUNet++ with Attention Mechanism

In [ ]:
class AttentionBlock(nn.Module):
    """
    Attention Block for focusing on relevant features
    """
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class ResidualBlock(nn.Module):
    """
    Residual Block with two convolutions
    """
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        residual = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += self.shortcut(residual)
        out = self.relu(out)
        
        return out


class ASPP(nn.Module):
    """
    Atrous Spatial Pyramid Pooling (ASPP) module
    """
    def __init__(self, in_channels, out_channels):
        super(ASPP, self).__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=6, dilation=6, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=18, dilation=18, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.conv_out = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        size = x.shape[2:]
        
        feat1 = self.conv1(x)
        feat2 = self.conv2(x)
        feat3 = self.conv3(x)
        feat4 = self.conv4(x)
        feat5 = F.interpolate(self.global_pool(x), size=size, mode='bilinear', align_corners=True)
        
        out = torch.cat([feat1, feat2, feat3, feat4, feat5], dim=1)
        out = self.conv_out(out)
        
        return out


class ResUNetPlusPlus(nn.Module):
    """
    ResUNet++ with Attention Mechanism for Steel Defect Detection
    """
    def __init__(self, in_channels=3, num_classes=4):
        super(ResUNetPlusPlus, self).__init__()
        
        # Encoder
        self.input_block = ResidualBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc1 = ResidualBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc2 = ResidualBlock(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc3 = ResidualBlock(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        
        # Bridge with ASPP
        self.bridge = ASPP(512, 1024)
        
        # Decoder with Attention
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.att1 = AttentionBlock(F_g=512, F_l=512, F_int=256)
        self.dec1 = ResidualBlock(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.att2 = AttentionBlock(F_g=256, F_l=256, F_int=128)
        self.dec2 = ResidualBlock(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att3 = AttentionBlock(F_g=128, F_l=128, F_int=64)
        self.dec3 = ResidualBlock(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att4 = AttentionBlock(F_g=64, F_l=64, F_int=32)
        self.dec4 = ResidualBlock(128, 64)
        
        # Output
        self.output = nn.Conv2d(64, num_classes, kernel_size=1)
    
    def forward(self, x):
        # Encoder
        x1 = self.input_block(x)  # 64 channels
        p1 = self.pool1(x1)
        
        x2 = self.enc1(p1)  # 128 channels
        p2 = self.pool2(x2)
        
        x3 = self.enc2(p2)  # 256 channels
        p3 = self.pool3(x3)
        
        x4 = self.enc3(p3)  # 512 channels
        p4 = self.pool4(x4)
        
        # Bridge
        bridge = self.bridge(p4)  # 1024 channels
        
        # Decoder
        d1 = self.up1(bridge)
        x4_att = self.att1(d1, x4)
        d1 = torch.cat([d1, x4_att], dim=1)
        d1 = self.dec1(d1)
        
        d2 = self.up2(d1)
        x3_att = self.att2(d2, x3)
        d2 = torch.cat([d2, x3_att], dim=1)
        d2 = self.dec2(d2)
        
        d3 = self.up3(d2)
        x2_att = self.att3(d3, x2)
        d3 = torch.cat([d3, x2_att], dim=1)
        d3 = self.dec3(d3)
        
        d4 = self.up4(d3)
        x1_att = self.att4(d4, x1)
        d4 = torch.cat([d4, x1_att], dim=1)
        d4 = self.dec4(d4)
        
        # Output
        output = self.output(d4)
        
        return output

print("✅ ResUNet++ with Attention model architecture defined!")

## 📐 Step 10: Define Loss Functions and Metrics

In [ ]:
class DiceLoss(nn.Module):
    """
    Dice Loss for segmentation tasks
    """
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        
        # Flatten
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        
        intersection = (pred_flat * target_flat).sum()
        dice = (2. * intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)
        
        return 1 - dice


class CombinedLoss(nn.Module):
    """
    Combined BCE and Dice Loss
    """
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super(CombinedLoss, self).__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    
    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        dice_loss = self.dice(pred, target)
        
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


def dice_coefficient(pred, target, smooth=1e-6):
    """
    Calculate Dice Coefficient (metric)
    """
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    
    intersection = (pred_flat * target_flat).sum()
    dice = (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
    
    return dice.item()


def iou_score(pred, target, smooth=1e-6):
    """
    Calculate Intersection over Union (IoU)
    """
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    
    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    
    iou = (intersection + smooth) / (union + smooth)
    
    return iou.item()

print("✅ Loss functions and metrics defined!")

## 🔄 Step 11: Split Data and Create DataLoaders

In [ ]:
# Split data into train and validation
train_df_split, val_df_split = train_test_split(
    train_df_processed, 
    test_size=0.2, 
    random_state=42,
    stratify=train_df_processed['HasDefect']
)

train_df_split = train_df_split.reset_index(drop=True)
val_df_split = val_df_split.reset_index(drop=True)

print(f"📊 Training samples: {len(train_df_split)}")
print(f"📊 Validation samples: {len(val_df_split)}")

# Create datasets
train_dataset = SteelDefectDataset(train_df_split, train_images_path, transform=train_transform)
val_dataset = SteelDefectDataset(val_df_split, train_images_path, transform=val_transform)

# Create dataloaders
BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"✅ DataLoaders created!")
print(f"📦 Training batches: {len(train_loader)}")
print(f"📦 Validation batches: {len(val_loader)}")

## 🏋️ Step 12: Training Setup

In [ ]:
# Initialize model
model = ResUNetPlusPlus(in_channels=3, num_classes=4).to(device)

# Loss and optimizer
criterion = CombinedLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

# Training parameters
NUM_EPOCHS = 30
BEST_VAL_LOSS = float('inf')

print(f"✅ Model initialized on {device}")
print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"📊 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 🚀 Step 13: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """
    Train for one epoch
    """
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    for batch_idx, (images, masks, _) in enumerate(loader):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Metrics
        running_loss += loss.item()
        running_dice += dice_coefficient(outputs, masks)
        running_iou += iou_score(outputs, masks)
        
        # Print progress
        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch [{batch_idx + 1}/{len(loader)}] - Loss: {loss.item():.4f}")
    
    epoch_loss = running_loss / len(loader)
    epoch_dice = running_dice / len(loader)
    epoch_iou = running_iou / len(loader)
    
    return epoch_loss, epoch_dice, epoch_iou


def validate_epoch(model, loader, criterion, device):
    """
    Validate for one epoch
    """
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            # Metrics
            running_loss += loss.item()
            running_dice += dice_coefficient(outputs, masks)
            running_iou += iou_score(outputs, masks)
    
    epoch_loss = running_loss / len(loader)
    epoch_dice = running_dice / len(loader)
    epoch_iou = running_iou / len(loader)
    
    return epoch_loss, epoch_dice, epoch_iou

print("✅ Training and validation functions defined!")

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_dice': [],
    'train_iou': [],
    'val_loss': [],
    'val_dice': [],
    'val_iou': []
}

print("🚀 Starting training...\n")

for epoch in range(NUM_EPOCHS):
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
    print("-" * 50)
    
    # Train
    train_loss, train_dice, train_iou = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_dice, val_iou = validate_epoch(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['train_iou'].append(train_iou)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)
    
    # Print metrics
    print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f} | Train IoU: {train_iou:.4f}")
    print(f"  Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")
    
    # Save best model
    if val_loss < BEST_VAL_LOSS:
        BEST_VAL_LOSS = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_dice': val_dice,
            'val_iou': val_iou
        }, 'best_resunet_model.pth')
        print(f"  ✅ Best model saved! (Val Loss: {val_loss:.4f})")
    
    print()

print("🎉 Training completed!")

## 📈 Step 14: Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Dice coefficient plot
axes[1].plot(history['train_dice'], label='Train Dice', marker='o')
axes[1].plot(history['val_dice'], label='Val Dice', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Coefficient')
axes[1].set_title('Training and Validation Dice Coefficient')
axes[1].legend()
axes[1].grid(True)

# IoU plot
axes[2].plot(history['train_iou'], label='Train IoU', marker='o')
axes[2].plot(history['val_iou'], label='Val IoU', marker='s')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('IoU Score')
axes[2].set_title('Training and Validation IoU')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 🎯 Step 15: Load Best Model for Inference

In [ ]:
# Load best model
checkpoint = torch.load('best_resunet_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✅ Best model loaded!")
print(f"📊 Best Val Loss: {checkpoint['val_loss']:.4f}")
print(f"📊 Best Val Dice: {checkpoint['val_dice']:.4f}")
print(f"📊 Best Val IoU: {checkpoint['val_iou']:.4f}")

## 🔮 Step 16: Prediction Function

In [ ]:
def predict_and_visualize(image_path, model, device, threshold=0.5):
    """
    Predict defects on a single image and visualize results
    
    Args:
        image_path: Path to the steel image
        model: Trained model
        device: Device to run inference on
        threshold: Confidence threshold for mask
    
    Returns:
        predictions: Dictionary with masks and confidence scores
    """
    # Load and preprocess image
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_size = image_rgb.shape[:2]
    
    # Apply transforms
    transform = A.Compose([
        A.Resize(256, 256),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    transformed = transform(image=image_rgb)
    image_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        output = model(image_tensor)
        output = torch.sigmoid(output)
    
    # Get predictions
    predictions = output.squeeze(0).cpu().numpy()
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Original image
    axes[0, 0].imshow(image_rgb)
    axes[0, 0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Color map for classes
    colors = [
        np.array([255, 0, 0]),      # Red for Class 1
        np.array([0, 255, 0]),      # Green for Class 2
        np.array([0, 0, 255]),      # Blue for Class 3
        np.array([255, 255, 0])     # Yellow for Class 4
    ]
    class_names = ['Class 1', 'Class 2', 'Class 3', 'Class 4']
    
    # Individual class predictions
    combined_overlay = image_rgb.copy()
    confidence_scores = []
    
    for i in range(4):
        mask = predictions[i]
        binary_mask = (mask > threshold).astype(np.uint8)
        
        # Resize mask to original size
        binary_mask_resized = cv2.resize(binary_mask, (original_size[1], original_size[0]), 
                                         interpolation=cv2.INTER_NEAREST)
        
        # Calculate confidence
        if binary_mask.sum() > 0:
            confidence = mask[binary_mask == 1].mean()
        else:
            confidence = 0.0
        confidence_scores.append(confidence)
        
        # Plot individual mask
        row = (i + 1) // 3
        col = (i + 1) % 3
        axes[row, col].imshow(binary_mask, cmap='gray')
        axes[row, col].set_title(f'{class_names[i]} (Conf: {confidence:.3f})', 
                                fontsize=12, fontweight='bold')
        axes[row, col].axis('off')
        
        # Add to combined overlay
        if binary_mask_resized.sum() > 0:
            color_mask = np.zeros_like(image_rgb)
            for c in range(3):
                color_mask[:, :, c] = binary_mask_resized * colors[i][c]
            combined_overlay = cv2.addWeighted(combined_overlay, 1, color_mask.astype(np.uint8), 0.4, 0)
    
    # Combined overlay
    axes[1, 2].imshow(combined_overlay)
    axes[1, 2].set_title('All Defects Overlay', fontsize=14, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "="*60)
    print("🔍 DEFECT DETECTION RESULTS")
    print("="*60)
    
    defects_found = False
    for i, conf in enumerate(confidence_scores):
        if conf > 0:
            print(f"✅ {class_names[i]}: Detected (Confidence: {conf:.3f})")
            defects_found = True
        else:
            print(f"❌ {class_names[i]}: Not detected")
    
    if not defects_found:
        print("\n✅ No defects detected - Steel surface appears clean!")
    else:
        print(f"\n⚠️  Overall Confidence Score: {np.mean([c for c in confidence_scores if c > 0]):.3f}")
    
    print("="*60)
    
    return {
        'masks': predictions,
        'confidence_scores': confidence_scores,
        'overlay': combined_overlay
    }

print("✅ Prediction function defined!")

## 🧪 Step 17: Test on Validation Samples

In [ ]:
# Test on random validation samples
print("🧪 Testing on validation samples...\n")

# Get samples with defects
defective_samples = val_df_split[val_df_split['HasDefect'] == True].sample(3)

for idx, row in defective_samples.iterrows():
    image_path = os.path.join(train_images_path, row['ImageId'])
    print(f"\n🖼️  Analyzing: {row['ImageId']}")
    results = predict_and_visualize(image_path, model, device)

## 📤 Step 18: Upload and Test Custom Image

In [ ]:
from google.colab import files
import io
from PIL import Image as PILImage

print("📤 Upload your steel image for defect detection!")
print("=" * 60)

# Upload file
uploaded = files.upload()

# Process uploaded image
for filename in uploaded.keys():
    print(f"\n🖼️  Processing uploaded image: {filename}")
    
    # Save uploaded file temporarily
    with open(filename, 'wb') as f:
        f.write(uploaded[filename])
    
    # Run prediction
    results = predict_and_visualize(filename, model, device, threshold=0.5)
    
    print(f"\n✅ Analysis complete for {filename}!")

## 💾 Step 19: Save Model and Export

In [ ]:
# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_architecture': 'ResUNetPlusPlus',
    'num_classes': 4,
    'input_channels': 3,
    'best_val_loss': checkpoint['val_loss'],
    'best_val_dice': checkpoint['val_dice'],
    'best_val_iou': checkpoint['val_iou']
}, 'steel_defect_resunet_final.pth')

print("✅ Final model saved as 'steel_defect_resunet_final.pth'")

# Download model
from google.colab import files
files.download('steel_defect_resunet_final.pth')
print("📥 Model downloaded!")

## 📊 Step 20: Model Summary and Performance Report

In [ ]:
print("="*80)
print("📊 STEEL DEFECT DETECTION MODEL - FINAL REPORT")
print("="*80)
print(f"\n🏗️  Model Architecture: ResUNet++ with Attention Mechanisms")
print(f"📦 Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"🎯 Number of Classes: 4")
print(f"\n📈 Best Performance Metrics:")
print(f"   • Validation Loss: {checkpoint['val_loss']:.4f}")
print(f"   • Validation Dice Score: {checkpoint['val_dice']:.4f}")
print(f"   • Validation IoU Score: {checkpoint['val_iou']:.4f}")
print(f"\n🔧 Training Configuration:")
print(f"   • Epochs: {NUM_EPOCHS}")
print(f"   • Batch Size: {BATCH_SIZE}")
print(f"   • Learning Rate: 1e-4")
print(f"   • Optimizer: Adam")
print(f"   • Loss Function: Combined BCE + Dice Loss")
print(f"\n📁 Dataset Information:")
print(f"   • Training Samples: {len(train_df_split)}")
print(f"   • Validation Samples: {len(val_df_split)}")
print(f"   • Image Size: 256x256")
print(f"\n✅ Model is ready for deployment!")
print("="*80)